<a href="https://colab.research.google.com/github/Alejooh/Taller-3/blob/main/simulacion_api_ml_alvarez_jose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autor: José Alejandro Alvaarez Sánchez
# Simulación DES: Infraestructura de una API de Machine Learning

Adaptación del modelo de eventos discretos visto en clase para evaluar la latencia (colas) y la gestión de créditos de nube (política s, Q) de una API de ML en producción.

In [ ]:
!pip install simpy


In [ ]:
import simpy
import random
import numpy as np
import scipy.stats as st

np.random.seed(30)
random.seed(30)


## 1. Parámetros del sistema

In [ ]:
# Colas (M/M/c)
LAMBDA = 30.0        # 30 peticiones por minuto
MU = 10.0            # 10 imágenes procesadas por minuto por nodo
NODOS_GPU = 4        # 4 nodos GPU (c)

# Inventario de créditos (política s, Q)
CREDITOS_INICIAL = 500
PUNTO_REORDEN_s = 50
RECARGA_Q = 400
LEAD_TIME = 2.0      # media de 2 minutos

# Simulación
TIEMPO_SIMULACION = 60   # 60 minutos continuos


## 2. Modelo de eventos discretos (SimPy)

In [ ]:
class InfraestructuraAPI:
    def __init__(self, env, num_nodos, creditos_ini, punto_reorden, cant_recarga, lead_time):
        self.env = env
        self.nodos_gpu = simpy.Resource(env, capacity=num_nodos)
        self.creditos = simpy.Container(env, init=creditos_ini, capacity=100000)
        self.punto_reorden = punto_reorden
        self.cant_recarga = cant_recarga
        self.lead_time = lead_time
        self.recargas_pendientes = 0
        self.tiempos_espera = []
        self.predicciones_ok = 0
        self.predicciones_fallidas = 0

    def solicitar_recarga(self):
        self.recargas_pendientes += 1
        tiempo_recarga = max(0.1, random.normalvariate(self.lead_time, 0.5))
        yield self.env.timeout(tiempo_recarga)
        yield self.creditos.put(self.cant_recarga)
        self.recargas_pendientes -= 1

    def controlar_creditos(self):
        while True:
            if self.creditos.level <= self.punto_reorden and self.recargas_pendientes == 0:
                self.env.process(self.solicitar_recarga())
            yield self.env.timeout(1)   # revisión cada 1 minuto

def peticion(env, infra, mu):
    llegada = env.now
    with infra.nodos_gpu.request() as solicitud:
        yield solicitud
        infra.tiempos_espera.append(env.now - llegada)
        yield env.timeout(random.expovariate(mu))
        if infra.creditos.level > 0:
            yield infra.creditos.get(1)
            infra.predicciones_ok += 1
        else:
            infra.predicciones_fallidas += 1

def generador_peticiones(env, infra, lam, mu):
    while True:
        yield env.timeout(random.expovariate(lam))
        env.process(peticion(env, infra, mu))


## 3. Evaluación con 30 réplicas

In [ ]:
def calcular_intervalo_confianza(datos, confianza=0.95):
    n = len(datos)
    media = np.mean(datos)
    h = st.sem(datos) * st.t.ppf((1 + confianza) / 2., n - 1)
    return media, media - h, media + h

def evaluacion_operacional(replicas, tiempo_sim, punto_reorden):
    resultados_wq = []
    resultados_fallidas = []
    for r in range(replicas):
        env = simpy.Environment()
        infra = InfraestructuraAPI(env, NODOS_GPU, CREDITOS_INICIAL, punto_reorden, RECARGA_Q, LEAD_TIME)
        env.process(generador_peticiones(env, infra, LAMBDA, MU))
        env.process(infra.controlar_creditos())
        env.run(until=tiempo_sim)
        resultados_wq.append(np.mean(infra.tiempos_espera) * 60)   # segundos
        resultados_fallidas.append(infra.predicciones_fallidas)

    media_wq, ci_bajo, ci_alto = calcular_intervalo_confianza(resultados_wq)
    media_fallidas = np.mean(resultados_fallidas)
    utilizacion = LAMBDA / (NODOS_GPU * MU)

    print(f"--- RESULTADOS ({replicas} RÉPLICAS, s = {punto_reorden}) ---")
    print(f"Wq promedio: {media_wq:.2f} segundos")
    print(f"IC 95% de Wq: [{ci_bajo:.2f} , {ci_alto:.2f}] segundos")
    print(f"Predicciones fallidas (promedio): {media_fallidas:.1f}")
    print(f"Utilización teórica de nodos GPU (ρ): {utilizacion:.1%}")

evaluacion_operacional(replicas=30, tiempo_sim=TIEMPO_SIMULACION, punto_reorden=PUNTO_REORDEN_s)


--- RESULTADOS (30 RÉPLICAS, s = 50) ---
Wq promedio: 3.03 segundos
IC 95% de Wq: [2.61 , 3.45] segundos
Predicciones fallidas (promedio): 91.9
Utilización teórica de nodos GPU (ρ): 75.0%


## 4. Análisis: fallas con utilización estable (Paso 6)

La utilización es ρ = λ/(c·μ) = 30/(4·10) = 75%, así que los nodos GPU no están saturados. Las fallas no vienen del cómputo sino del inventario de créditos: cada predicción consume un crédito a ~30/min. La demanda durante el Lead Time es λ·LT = 30·2 = 60 créditos, mayor que el punto de reorden s = 50. Cuando se dispara la recarga al llegar a 50, durante los 2 minutos de espera se consumen ~60 créditos y el stock se agota antes de que lleguen los 400 nuevos. La revisión cada minuto y la variabilidad de Poisson agravan el quiebre. En resumen: el hardware tiene capacidad de sobra, pero el sistema se queda sin tokens.

## 5. Calibración de s para cero fallas (Paso 7)

Para cubrir el ciclo de reabastecimiento: demanda en Lead Time (λ·LT = 60) + demanda durante la revisión (λ·1 = 30) ≈ 90 créditos, más ~50 de stock de seguridad por la ráfaga de Poisson y la variación del Lead Time → **s = 140**. Verificado por barrido sobre 200 réplicas (el mínimo sin fallas es ≈134; se usa 140 por margen). Q = 400 y stock inicial = 500 se mantienen. Se reconfigura s = 140 y se re-ejecutan las 30 réplicas para confirmar cero predicciones fallidas.

In [ ]:
PUNTO_REORDEN_s = 140
evaluacion_operacional(replicas=30, tiempo_sim=TIEMPO_SIMULACION, punto_reorden=PUNTO_REORDEN_s)


--- RESULTADOS (30 RÉPLICAS, s = 140) ---
Wq promedio: 3.05 segundos
IC 95% de Wq: [2.66 , 3.45] segundos
Predicciones fallidas (promedio): 0.0
Utilización teórica de nodos GPU (ρ): 75.0%
